## Week 1 — PySpark Transformations Practice
Exploring the Garmin activities Delta table using core Spark DataFrame operations.

In [0]:
df = spark.table("garmin_lakehouse.raw.activities")
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")

In [0]:
df_selected = df.select("activityId",
    "activityName", 
    "startTimeLocal",
    "activityType",
    "distance",
    "duration",
    "averageSpeed",
    "averageHR",
    "maxHR",
    "calories")

df_selected.display()

In [0]:
# FILTER - only running activities
from pyspark.sql.functions import col

df_runs = df_selected.filter(col("activityType").contains("running"))

print(f"Total activities: {df_selected.count()}")
print(f"Running only: {df_runs.count()}")
df_runs.display()

In [0]:
# GROUPBY - count of each activity type
from pyspark.sql.functions import count

df_by_type = df_selected \
    .groupBy("activityType").agg(count("activityId").alias("activity_count")).sort(col("activity_count").desc())

df_by_type.display()

In [0]:
# WITHCOLUMN - add derived columns
from pyspark.sql.functions import col, round, to_timestamp

df_enriched = df_runs \
    .withColumn("distance_km", round(col("distance") / 1000, 2)) \
    .withColumn("duration_min", round(col("duration") / 60, 2)) \
    .withColumn("pace_min_per_km", round(col("duration_min") / col("distance_km"), 2)) \
    .withColumn("start_timestamp", to_timestamp(col("startTimeLocal")))

df_enriched.select(
    "activityName",
    "start_timestamp",
    "distance_km",
    "duration_min",
    "pace_min_per_km",
    "averageHR"
).display()

In [0]:
# SORT - find your best runs by pace
from pyspark.sql.functions import col, round

df_best_runs = df_enriched \
    .filter(col("distance_km") > 5) \
    .select(
        "activityName",
        "start_timestamp",
        "distance_km",
        "duration_min",
        "pace_min_per_km",
        "averageHR"
    ) \
    .sort(col("pace_min_per_km").asc())

df_best_runs.display()